# Linear Fitting

<div class="alert alert-block alert-info">
<strong>Questions:</strong>

<ul>
    <li>How can I use Python to fit a linear model?</li>
</ul>

<strong>Objectives:</strong>

<ul>
    <li>Fit a linear model using statsmodels</li>
</ul>
</div>


A common exercise in introductory chemistry is fitting a linear model to absorbance measurements at a particular wavelength vs. concentration of an analyte using spectrophotometry. 
In these cases, the absorbance of a sample is described using [Beer-Lambert Law](https://en.wikipedia.org/wiki/Beer%E2%80%93Lambert_law)

$$
A = \varepsilon \cdot l \cdot c
$$

where $A$ is the absorbance, $\varepsilon$ is the absorptivity constant, $l$ is path length, and $c$ is the concentration of the sample.


In this notebook, we will read in absorbance data, fit a linear model using a Python library called [statsmodels](https://www.statsmodels.org/stable/index.html), and calculate the concentrations of unknowns using our model.
We will utilize skills learned in our previous lessons, including reading, accessing, and analyzing data using `pandas` and visualizing data using `plotly`.
We will add a new skill - fitting a linear model with a Python library called `statsmodels`. 

## Importing Libraries and Reading Data

Note that when fitting a model in Python, there are a number of options for the library you might pick including NumPy, SciPy, SciKit-Learn, and statsmodels.
The library that you pick might be based on personal preference or features that a particular library offers.
In this notebook, we show fitting with `statsmodels` because of the ease of seeing fit statics and defining a formula for fitting.

For `statsmodels`, we will use the formula API. 
This will allow us to define the relationship we'd like to fit using a formula as a string.

To start this notebook, we will import the libraries we need. 
We will import the following:

Library Name | Purpose
-------------|--------
[pandas](https://pandas.pydata.org/docs/)       | reading and processing data
[plotly](https://plotly.com/python/getting-started/)       | interactive plots and visualization
[statsmodels](https://www.statsmodels.org/stable/index.html)  | data fitting


The cell below imports the libraries we will use for our analysis. 

**Instructor edition.** Completed examples and worked solutions. Run all cells from top to bottom using the project Python environment. Data are read from the parent repository.

In [1]:
from pathlib import Path
import sys

# Support kernels started in the repository root or in instructor-notebooks.
ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "data").is_dir():
    raise FileNotFoundError("Open this notebook from the repository or instructor-notebooks folder.")
sys.path.insert(0, str(ROOT))

In [2]:
# imports 
import pandas as pd # for reading data from a file and putting in a table
import plotly.express as px # for plotting

import statsmodels.formula.api as smf # for fitting

For the example, we will use data stored in the file `data/protein_assay.csv`.
This data represents absorbance data recorded in a Bradford Assay for determining protein concentration.

After we have imported our libraries, we will next use `pandas` to read in our data.
Our data is stored in a comma separated value (CSV) file, though pandas can also read from Excel files.

In [3]:
# Read in file
df = pd.read_csv(ROOT / "data/protein_assay.csv")

# View the first five rows.
df.head()

,concentration,absorbance
0,0.2,0.285
1,0.4,0.485
2,0.6,0.621
3,0.8,0.799
4,1.0,1.010


### Visualizing Data

After reading in our data, we might wish to inspect it visually. 
We can do this using plotly express (imported as `px`).

In [4]:
# Create a figure using px.scatter.
fig = px.scatter(df, x='concentration', y="absorbance")

# Show the figure.
fig.show()

Note that you can change the axis labels using the following syntax.

In [5]:
# Create a figure using px.scatter.
fig = px.scatter(df, x='concentration', y="absorbance", 
                 labels={"concentration":"Concentration (mg/mL)", "absorbance":"Absorbance at 595 nm"})

# Show the figure.
fig.show()

## Fitting a Linear Model

To fit a linear equation to our data, we can use a library called [statsmodels](https://www.statsmodels.org/stable/index.html). 
`statsmodels` is a Python module that provides classes and functions for the estimation of many different statistical models, as well as for conducting statistical tests, and statistical data exploration.

While there are numerous options for fitting data in Python, `statsmodels` is particularly beneficial when dealing with linear models. 
Note that this encompasses more than just "linear equations". 
This library is especially handy when you need straightforward access to fit parameters and in-depth statistical metrics related to the fit.

In particular, we are using a part of `statsmodels` called the formula API.
The formula API lets you define a formula as a string for fitting and is specifically designed to work with dataframes.
When defining a formula, you use the column names in a string to define the relationship.
Note that the column names should not have spaces, or entering the relationship is a bit more complicated.
For example, if we had data representing pressure and temperature at constant volume and expected them to follow a linear relationship such as for an ideal gas

$$
P = \frac{n R T}{V}
$$

we would write `"P ~ T"` when using the `statsmodels` formula API. 
This is assuming that our dataframe has columns named `P` and `T`


As an optional extension, a quadratic model could use the formula `"y ~ np.power(x, 2) + x"`. That example requires `import numpy as np` in the same notebook before fitting. NumPy is not needed for the linear fit below. 

To use the formula API, we will use `smf` (imported in first cell). 
We will use ordinary least squares (`ols`) for our fit, though [a number of other options are offered](https://www.statsmodels.org/dev/api.html#statsmodels-formula-api).


In [6]:
regression = smf.ols("absorbance ~ concentration", data=df).fit()

We can see a summary of the fit including the `R-squared` by using the `.summary()` method.

In [7]:
regression.summary()

/home/jessica-nash/teaching/python-experimental-chem/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:3158: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  omni, omnipv = cached("omni_normtest", lambda: omni_normtest(self.wresid))


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             absorbance   R-squared:                       0.995
Model:                            OLS   Adj. R-squared:                  0.993
Method:                 Least Squares   F-statistic:                     749.4
Date:                Tue, 29 Sep 2026   Prob (F-statistic):           1.06e-05
Time:                        21:12:36   Log-Likelihood:                 14.638
No. Observations:                   6   AIC:                            -25.28
Df Residuals:                       4   BIC:                            -25.69
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.1279      0.024      5.316      0.006       0.061       0.195
concentration     0.8454      0.031     27.374      0.000       0.760       0.931
==============================================================================
Omnibus:                          nan   Durbin-Watson:                   2.852
Prob(Omnibus):                    nan   Jarque-Bera (JB):                0.692
Skew:                           0.659   Prob(JB):                        0.708
Kurtosis:                       1.984   Cond. No.                         4.48
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

If we would like to force the model to not have an intercept, we use a special `statsmodel` syntax. 
Adding `-1` to our formula forces the intercept to be 0.

In [8]:
# force the intercept to be 0
regression = smf.ols("absorbance ~ concentration - 1", data=df).fit()
regression.summary()

/home/jessica-nash/teaching/python-experimental-chem/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:3158: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  omni, omnipv = cached("omni_normtest", lambda: omni_normtest(self.wresid))


<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:             absorbance   R-squared (uncentered):                   0.994
Model:                            OLS   Adj. R-squared (uncentered):              0.993
Method:                 Least Squares   F-statistic:                              833.3
Date:                Tue, 29 Sep 2026   Prob (F-statistic):                    9.35e-07
Time:                        21:12:36   Log-Likelihood:                          8.3757
No. Observations:                   6   AIC:                                     -14.75
Df Residuals:                       5   BIC:                                     -14.96
Df Model:                           1                                                  
Covariance Type:            nonrobust                                                  
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
concentration     0.9930      0.034     28.866      0.000       0.905       1.081
==============================================================================
Omnibus:                          nan   Durbin-Watson:                   0.590
Prob(Omnibus):                    nan   Jarque-Bera (JB):                0.344
Skew:                          -0.475   Prob(JB):                        0.842
Kurtosis:                       2.311   Cond. No.                         1.00
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

The model parameters are in `.params` of the fit variable.

In [9]:
regression.params

concentration    0.992967
dtype: float64

To get the slope, we will get the coefficient in front of the concentration variable.

In [10]:
slope = regression.params["concentration"]
print(slope)

0.9929670329670331


We can see what our model predicts for our input concentration values by using the `predict` method.
In the cell below, we save the results in a new column in our dataframe.

In [11]:
df["predicted"] = regression.predict()

In [12]:
df.head()

,concentration,absorbance,predicted
0,0.2,0.285,0.198593
1,0.4,0.485,0.397187
2,0.6,0.621,0.595780
3,0.8,0.799,0.794374
4,1.0,1.010,0.992967


In [13]:
fig = px.scatter(x= df["concentration"], y=df["absorbance"])
fig.add_scatter(x=df["concentration"], y=df["predicted"], mode="lines", name="model")
fig.show()

## Fitting unknowns

Now that we have our model and slope, we can use it to calculate the protein concentrations for a set of unknowns.

<div class="alert alert-block alert-warning">
<strong>Exercise</strong>

<p>
Use pandas to read in the file <code>data/protein_samples.csv</code>.
</p>

<p>
Next, use our calculated slope to predict concentration based on measured absorbance.
</p>
</div>


**Solution reasoning:** The current model fixes the intercept to zero, so concentration = absorbance / slope, in mg/mL. With a fitted intercept, use (absorbance − intercept) / slope. A zero intercept is an assumption to justify from the experiment; check estimated concentrations against the calibration range.

In [14]:
samples = pd.read_csv(ROOT / "data/protein_samples.csv")
samples.head()

,Sample #,absorbance
0,1,0.183
1,2,0.682
2,3,0.759
3,4,1.340
4,5,0.935


In [15]:
samples["concentration"] = samples["absorbance"] / slope
samples

,Sample #,absorbance,concentration
0,1,0.183,0.184296
1,2,0.682,0.686830
2,3,0.759,0.764376
3,4,1.340,1.349491
4,5,0.935,0.941622
5,6,1.013,1.020175


In [16]:
print("Calibration slope:", slope)
print("Calibration concentration range:", df["concentration"].min(), df["concentration"].max())

Calibration slope: 0.9929670329670331
Calibration concentration range: 0.2 1.2


In [17]:
samples["within_calibration_range"] = samples["concentration"].between(df["concentration"].min(), df["concentration"].max())
samples

,Sample #,absorbance,concentration,within_calibration_range
0,1,0.183,0.184296,False
1,2,0.682,0.686830,True
2,3,0.759,0.764376,True
3,4,1.340,1.349491,False
4,5,0.935,0.941622,True
5,6,1.013,1.020175,True


## Exercise - Perform a Linear Fit

<div class="alert alert-block alert-warning">
<strong>Final Exercise: Create Your Own Notebook</strong>

<p>
<strong>Create a new notebook</strong> named <code>04-groundwater-exercise.ipynb</code> in the same folder as the lesson notebooks. Complete this exercise there.
</p>
<p>
Start with a Markdown cell containing a title and a brief description of the problem. Add code cells for your setup and analysis. Import pandas and the statsmodels formula API, then load <code>data/ground_water.csv</code> into a new DataFrame. Inspect its columns: <code>pH</code> is the dependent variable and <code>Bicarb</code> is the independent variable. Fit a new model and display its regression summary.
</p>
<p>
Each notebook needs its own setup: variables and imports from the lesson notebook are not automatically available in your new notebook.
</p>


<p>
Now that you have completed a simple linear regression exercise with protein assay data, here is a problem with a slightly larger dataset, taken from a ground water survey of wells in Texas kindly provided by Houghton-Mifflin. 
The data for this exercise is in the file <code>data/ground_water.csv</code>.
Using the skills you have learned with pandas and <code>statsmodels</code>, get the linear regression statistics for the relationship between pH (dependent variable) and bicarbonate levels (ppm in well water in Texas; independent variable).
</p>

<p>
Finish with a Markdown cell explaining your results. Save your notebook, restart its kernel, and run all cells from top to bottom. Check that it completes without errors and reproduces your results. This checks that every step needed for your analysis is included in the notebook.
</p>
</div>





**Worked solution:** [04-groundwater-exercise.ipynb](04-groundwater-exercise.ipynb). This separate notebook includes all required setup and interpretation.